# 1.Load dataset 

In [1]:
import pandas as pd
df = pd.read_csv('bbc_news.csv')
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(df.isnull().sum())

Total rows: 2224
Total columns: 2
Category    0
Text        0
dtype: int64


In [2]:
# Percentage of missing values per column
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage)
# Rows with any missing values
missing_rows = df[df.isnull().any(axis=1)]
print(missing_rows)


Category    0.0
Text        0.0
dtype: float64
Empty DataFrame
Columns: [Category, Text]
Index: []


In [3]:
# Summary of the dataset
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2224 entries, 0 to 2223
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  2224 non-null   object
 1   Text      2224 non-null   object
dtypes: object(2)
memory usage: 34.9+ KB
None


## 1.1. Preprocessing data

In [4]:
texts = df['Text']
labels = df['Category']

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(texts).toarray()
X

array([[0.02534578, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.00626441, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [6]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(labels) 
y

array([0, 0, 0, ..., 4, 4, 4])

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state=42)


In [8]:
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Dropout
model = Sequential([
    Dense(128,activation='relu',input_dim=X_train.shape[1]),
    Dropout(0.5),
    Dense(64,activation='relu'),
    Dropout(0.5),
    Dense(len(set(y)),activation='softmax')
])

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

model.fit(X_train,y_train,epochs=10,batch_size=32,validation_data=(X_test,y_test))

loss,accuracy = model.evaluate(X_test,y_test)
print(f'Accuracy:{accuracy}')

c:\Users\Sreylen\miniconda3\envs\lenenv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.3493 - loss: 1.5501 - val_accuracy: 0.9011 - val_loss: 1.0120
Epoch 2/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8446 - loss: 0.8086 - val_accuracy: 0.9640 - val_loss: 0.2376
Epoch 3/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9713 - loss: 0.2188 - val_accuracy: 0.9708 - val_loss: 0.1254
Epoch 4/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9814 - loss: 0.1098 - val_accuracy: 0.9708 - val_loss: 0.1052
Epoch 5/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9932 - loss: 0.0525 - val_accuracy: 0.9685 - val_loss: 0.1021
Epoch 6/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9930 - loss: 0.0505 - val_accuracy: 0.9640 - val_loss: 0.1005
Epoch 7/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9983 - loss: 0.0274 - val_accuracy: 0.9685 - val_loss: 0.1037
Epoch 8/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9994 - loss: 0.0193 - val_accuracy: 0.9640 - v

In [10]:
import pickle
model_json  =model.to_json()
with open('model_architecture.pkl', 'wb') as file:
    pickle.dump(model_json, file)
print("Model architecture saved to 'model_architecture.pkl'")

Model architecture saved to 'model_architecture.pkl'


In [11]:
model_and_vectorizer = {
    'model': model,         # The trained model object
    'vectorizer': vectorizer  # The vectorizer object
}

# Save the dictionary to a file using pickle
with open('model_and_vectorizer.pkl', 'wb') as f:
    pickle.dump(model_and_vectorizer, f)

print("Model and Vectorizer have been saved.")

Model and Vectorizer have been saved.


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# New document
new_document = ['The stock market is experiencing significant growth today.']

# Preprocess the document (use the same `vectorizer` as during training)
X_new = vectorizer.transform(new_document)

# Predict with the model
predicted_probs = model.predict(X_new)
predicted_class = np.argmax(predicted_probs)

# Class mapping
categories = ['business', 'entertainment', 'politics', 'sport', 'tech']  # Example categories
print(f'Predicted category: {categories[predicted_class]}')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
Predicted category: business


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# New document
new_document = ['For the first time, Claxton has only been preparing for a campaign over the hurdles - which could explain her leap in form. In previous seasons, the 25-year-old also contested the long jump but since moving from Colchester to London she has re-focused her attentions. Claxton will see if her new training regime pays dividends at the European Indoors which take place on 5-6 March.']

# Preprocess the document (use the same `vectorizer` as during training)
X_new = vectorizer.transform(new_document)

# Predict with the model
predicted_probs = model.predict(X_new)
predicted_class = np.argmax(predicted_probs)

# Class mapping
categories = ['business', 'entertainment', 'politics', 'sport', 'tech']  # Example categories
print(f'Predicted category: {categories[predicted_class]}')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
Predicted category: sport


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# New document
new_document =['The “Find Song by Lyrics” (or partial lyrics)  tool can help you figure it out and solve your earworm. It’s simple—no artist name required. Just type the few lyrics you know, and once you’re finished entering them, our tool will help identify potential song matches. Don’t worry, you don’t need perfect lyrics to use this tool.'] 
# Preprocess the document (use the same `vectorizer` as during training)
X_new = vectorizer.transform(new_document)

# Predict with the model
predicted_probs = model.predict(X_new)
predicted_class = np.argmax(predicted_probs)

# Class mapping
categories = ['business', 'entertainment', 'politics', 'sport', 'tech']  # Example categories
print(f'Predicted category: {categories[predicted_class]}')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
Predicted category: tech
